# TLS processor walkthrough

This notebook demonstrates analysis-ready processing of the Ankasa TLS mesh and tree model. File parsing remains the responsibility of `TLSProvider`; `TLSProcessor` validates those datasets and performs spatial preparation and structural summaries.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from tomo_canopy.processors import TLSProcessor

## Locate the TLS sources

Set `CRS` from authoritative source metadata before integrating these coordinates with another sensor.

In [ ]:
project_root = next(
    path
    for path in (Path.cwd(), Path.cwd().parent)
    if (path / "pyproject.toml").exists()
)
data_dir = project_root / "data" / "TLS" / "Ankasa"
mesh_path = data_dir / "c01_r02_subsampled_1cm_trees_mesh.ply"
trees_path = data_dir / "c01_r02_subsampled_1cm_trees.txt"
CRS = None  # Example: "EPSG:32630"; verify this from source metadata.

assert mesh_path.exists(), f"Missing {mesh_path}"
assert trees_path.exists(), f"Missing {trees_path}"

## Open and validate the mesh

`TLSProcessor.open` delegates reading to its provider and validates the resulting geometry. Binary vertex and face arrays remain memory-mapped.

In [ ]:
processor = TLSProcessor()
mesh = processor.open(mesh_path, crs=CRS)
processor.validate(mesh, check_finite=True)
mesh

## Spatial bounds and local coordinates

Local coordinates improve numerical stability and make plots easier to read. The processor records the global origin in dataset attributes.

In [ ]:
bounds = processor.bounds(mesh)
bounds

In [ ]:
local_mesh = processor.to_local_coordinates(mesh)
print(
    "Local origin:",
    local_mesh.attrs["coordinate_origin_x"],
    local_mesh.attrs["coordinate_origin_y"],
    local_mesh.attrs["coordinate_origin_z"],
)

## Reproducible point-cloud preview

A sampled set of mesh vertices is suitable for a quick Matplotlib preview. Sampling removes face connectivity because the selected vertices no longer form the original surface.

In [ ]:
preview = processor.sample_vertices(local_mesh, count=100_000, seed=42)
preview

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(projection="3d")
points = ax.scatter(
    preview.x,
    preview.y,
    preview.z,
    c=preview.z,
    cmap="viridis",
    s=0.2,
)
ax.set(xlabel="Local x [m]", ylabel="Local y [m]", zlabel="Local z [m]")
fig.colorbar(points, ax=ax, label="Local height [m]", shrink=0.7)
plt.show()

## Process tree structure

The tree model retains radii and parent relationships. `tree_metrics` summarizes each reconstructed tree.

In [ ]:
trees = processor.open(trees_path, crs=CRS)
metrics = processor.tree_metrics(trees)
metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(metrics.height, bins=40, color="forestgreen")
axes[0].set(xlabel="Tree height [m]", ylabel="Tree count")
axes[0].set_title("Height distribution")

axes[1].scatter(metrics.height, metrics.maximum_radius, s=5, alpha=0.5)
axes[1].set(xlabel="Tree height [m]", ylabel="Maximum radius [m]")
axes[1].set_title("Tree dimensions")
plt.tight_layout()
plt.show()

## Plot an individual tree

Each non-root segment is connected to its local `parent_id`. Line width is scaled by segment radius.

In [ ]:
selected_tree_id = 0
tree_indices = np.flatnonzero(trees.tree_id.data == selected_tree_id)
tree = processor.to_local_coordinates(trees.isel(segment=tree_indices))

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(projection="3d")
for segment_index in range(tree.sizes["segment"]):
    parent_index = int(tree.parent_id.data[segment_index])
    if parent_index < 0:
        continue
    ax.plot(
        [tree.x.data[parent_index], tree.x.data[segment_index]],
        [tree.y.data[parent_index], tree.y.data[segment_index]],
        [tree.z.data[parent_index], tree.z.data[segment_index]],
        color="saddlebrown",
        linewidth=max(0.25, float(tree.radius.data[segment_index]) * 20),
    )
ax.set(xlabel="Local x [m]", ylabel="Local y [m]", zlabel="Local z [m]")
ax.set_title(f"TLS tree {selected_tree_id}")
plt.show()